# plan
### read lines 
### parse each line 
### extract: query, x, y, etc
### create pandas data frame 

In [1]:
import json
import pandas as pd

In [3]:
df = pd.read_json("../../data/example_1k.jsonl", lines=True)

display(df.head())

,latitude,longitude,message_id,model_response_full,model_response_timestamp,prod_classifier_agent,query,region_id,region_name
0,37.874904,32.492129,7475f82f-d35f-411c-876b-a233f6cd9d3c,"[None, [9ef9829a-35d6-4d26-a2d2-1e44e3adc2c1],...",1771686554,Geo Agent,Konumumdan en yakın rotayı çıkar,101474,Konya
1,41.011219,28.978176,2c53e7c8-ae12-462a-96fe-9c5fdbace8c4,"[None, [c007eac2-d98c-4ddc-818f-2432086ebaa7],...",1772400724,Geo Agent,Takı Dünyası Nerdedir?,11508,Istanbul
2,41.011219,28.978176,0eba2ebf-c5e5-4287-bed0-b76882acda23,"[None, [57178adf-d460-41aa-85f2-d8492219f12d],...",1771065812,Geo Agent,Şuan bulunduğum konumdan bursa görükle ye gide...,11508,Istanbul
3,37.062688,37.379510,ea56cc31-135c-42ca-9f29-0dd20aa4a56d,"[None, [31955092-3872-4116-b726-a8f428e5119a],...",1771742482,Geo Agent,Milano da gezilecek yerler bana söyler misin?,103825,Gaziantep
4,37.181190,33.222247,c12cf199-14c5-47b5-8ca8-7bc258300c38,"[None, [9a1e0b68-4d5a-45cf-a140-7b8c2c575aa1],...",1771836177,Geo Agent,karaman adana arası kaç km ve tır ne yakar,103701,Karaman


In [8]:
# Берем значение колонки model_response_full из первой строки
sample_response = df.loc[0, "model_response_full"]

print(f"Тип объекта: {type(sample_response)}")
print(f"Длина списка: {len(sample_response)}\n")

# Посмотрим, что лежит под каждым индексом
for i, item in enumerate(sample_response):
    # Ограничим вывод 100 символами, чтобы не засорять экран
    item_str = str(item)[:100] + "..." if len(str(item)) > 100 else str(item)
    print(f"Индекс {i} | Тип: {type(item).__name__} | Значение: {item_str}")

Тип объекта: <class 'list'>
Длина списка: 9

Индекс 0 | Тип: NoneType | Значение: None
Индекс 1 | Тип: list | Значение: ['9ef9829a-35d6-4d26-a2d2-1e44e3adc2c1']
Индекс 2 | Тип: list | Значение: ['7475f82f-d35f-411c-876b-a233f6cd9d3c']
Индекс 3 | Тип: list | Значение: ['default']
Индекс 4 | Тип: list | Значение: ['{"IsCompleteResults":true,"IsFinalMarkdownText":true,"LinksData":[{"Description":"Bts Rota Khimki ...
Индекс 5 | Тип: list | Значение: [[]]
Индекс 6 | Тип: NoneType | Значение: None
Индекс 7 | Тип: NoneType | Значение: None
Индекс 8 | Тип: list | Значение: [[{'content': 'Konumumdan en yakın rotayı çıkar ', 'extra_info': {'timestamp_seconds': '1771686550'}...


In [9]:
import json

# 1. Функция для извлечения Финального Текста (из индекса 4)
def get_final_text(response_list):
    try:
        # Проверяем, что это список и нужный индекс существует
        if isinstance(response_list, list) and len(response_list) > 4:
            payload_str = response_list[4][0]
            
            # Парсим JSON-строку
            data = json.loads(payload_str)
            
            # Достаем ключ TargetMarkdownText
            return data.get("TargetMarkdownText", None)
    except Exception:
        return None

# 2. Функция для извлечения ссылок на источники (из индекса 4)
def get_source_links(response_list):
    try:
        if isinstance(response_list, list) and len(response_list) > 4:
            payload_str = response_list[4][0]
            data = json.loads(payload_str)
            
            # Достаем список ссылок LinksData
            links_data = data.get("LinksData", [])
            
            # Вытаскиваем только сами URL адреса в виде списка
            urls = [link.get("FullUrl") for link in links_data if "FullUrl" in link]
            return urls
    except Exception:
        return []

# 3. Функция для анализа работы агента (из индекса 8)
# Например, посчитаем, сколько сообщений/шагов понадобилось для ответа
def get_agent_steps_count(response_list):
    try:
        if isinstance(response_list, list) and len(response_list) > 8:
            dialog_history = response_list[8][0]
            return len(dialog_history)
    except Exception:
        return 0

# --- ПРИМЕНЯЕМ ФУНКЦИИ К ДАТАФРЕЙМУ ---

# Создаем новые чистые колонки
df['final_answer'] = df['model_response_full'].apply(get_final_text)
df['source_urls'] = df['model_response_full'].apply(get_source_links)
df['steps_count'] = df['model_response_full'].apply(get_agent_steps_count)

# Смотрим на красоту (выведем только нужные колонки)
display(df[['query', 'steps_count', 'source_urls', 'final_answer']].head())

,query,steps_count,source_urls,final_answer
0,Konumumdan en yakın rotayı çıkar,4,"[https://www.cybo.com/RU-biz/bts-rota, https:/...","<media type=""organics_list"" tag=""quick""></medi..."
1,Takı Dünyası Nerdedir?,7,[https://yandex.com.tr/maps/org/taki_dunyasi/1...,"<media type=""organics_list"" tag=""quick""></medi..."
2,Şuan bulunduğum konumdan bursa görükle ye gide...,2,[],"<media type=""organics_list"" tag=""quick""></medi..."
3,Milano da gezilecek yerler bana söyler misin?,2,[],"<media type=""organics_list"" tag=""quick""></medi..."
4,karaman adana arası kaç km ve tır ne yakar,6,[https://yandex.com.tr/maps/983/turkey/routes/...,"<media type=""organics_list"" tag=""quick""></medi..."


In [10]:
def get_organizations(response_list):
    try:
        # Проверяем структуру (как делали раньше)
        if isinstance(response_list, list) and len(response_list) > 4:
            payload_str = response_list[4][0]
            data = json.loads(payload_str)
            
            # Идем по пути: StructuredData -> BlocksData
            blocks = data.get("StructuredData", {}).get("BlocksData", [])
            
            orgs_found = []
            
            # Перебираем все блоки ответа
            for block in blocks:
                # Ищем блок, который содержит список организаций
                if block.get("ContentType") == "OrganizationList":
                    # Достаем сам список
                    organizations = block.get("OrganizationList", {}).get("Organizations", [])
                    
                    # Проходимся по каждой организации и берем ее название (title)
                    for org in organizations:
                        title = org.get("title")
                        # Можно также достать адрес: address = org.get("address")
                        if title:
                            orgs_found.append(title)
                            
            return orgs_found # Возвращаем список найденных названий
            
    except Exception:
        pass
    
    return [] # Если ничего не нашли, возвращаем пустой список

# Применяем новую функцию
df['organizations'] = df['model_response_full'].apply(get_organizations)

# Посмотрим на результат!
display(df[['query', 'organizations', 'final_answer']].head())

,query,organizations,final_answer
0,Konumumdan en yakın rotayı çıkar,[Konya Büyükşehir Belediye Stadyumu],"<media type=""organics_list"" tag=""quick""></medi..."
1,Takı Dünyası Nerdedir?,"[Takı Dünyası, Takı Dünyası]","<media type=""organics_list"" tag=""quick""></medi..."
2,Şuan bulunduğum konumdan bursa görükle ye gide...,[],"<media type=""organics_list"" tag=""quick""></medi..."
3,Milano da gezilecek yerler bana söyler misin?,[],"<media type=""organics_list"" tag=""quick""></medi..."
4,karaman adana arası kaç km ve tır ne yakar,[],"<media type=""organics_list"" tag=""quick""></medi..."


In [11]:
import re

def clean_html_tags(text):
    if not isinstance(text, str):
        return text
    # Удаляем все, что находится внутри < ... >
    clean_text = re.sub(r'<[^>]+>', '', text)
    # Удаляем лишние переносы строк
    return clean_text.strip()

# Применяем очистку к нашей колонке
df['clean_answer'] = df['final_answer'].apply(clean_html_tags)

display(df[['query', 'organizations', 'clean_answer']].head())

,query,organizations,clean_answer
0,Konumumdan en yakın rotayı çıkar,[Konya Büyükşehir Belediye Stadyumu],Araçla Konyaspor Stadyumu’na en yakın rota ve ...
1,Takı Dünyası Nerdedir?,"[Takı Dünyası, Takı Dünyası]",İstanbul'da iki farklı Takı Dünyası mağazası b...
2,Şuan bulunduğum konumdan bursa görükle ye gide...,[],Could you clarify what you’re looking for? I c...
3,Milano da gezilecek yerler bana söyler misin?,[],
4,karaman adana arası kaç km ve tır ne yakar,[],Karaman’dan Adana’ya tır ile ulaşım için **en ...


In [12]:
import json
import pandas as pd

def extract_all_organizations(df):
    """
    Проходит по всему DataFrame, извлекает все организации 
    и возвращает новый "плоский" DataFrame.
    """
    all_orgs = []
    
    # Проходим по каждой строке (каждому запросу) с помощью iterrows()
    for index, row in df.iterrows():
        msg_id = row['message_id']
        response_list = row['model_response_full']
        
        try:
            # Идем к структурам данных
            if isinstance(response_list, list) and len(response_list) > 4:
                payload_str = response_list[4][0]
                data = json.loads(payload_str)
                blocks = data.get("StructuredData", {}).get("BlocksData", [])
                
                # Перебираем блоки в поисках OrganizationList
                for block in blocks:
                    if block.get("ContentType") == "OrganizationList":
                        organizations = block.get("OrganizationList", {}).get("Organizations", [])
                        
                        # Перебираем каждую найденную организацию
                        for org in organizations:
                            
                            # Безопасное извлечение рейтинга (иногда его нет)
                            rating_val = None
                            if "rating" in org and isinstance(org["rating"], dict):
                                rating_val = org["rating"].get("rating")
                            
                            # Безопасное извлечение координат
                            lat, lon = None, None
                            if "coords" in org and isinstance(org["coords"], dict):
                                lat = org["coords"].get("lat")
                                lon = org["coords"].get("long") # Обрати внимание, там ключ "long"
                                
                            # Собираем данные в плоский словарь
                            org_record = {
                                "message_id": msg_id, # Связь с основным запросом!
                                "org_id": org.get("id"),
                                "name": org.get("title"),
                                "address": org.get("address"),
                                "rating": rating_val,
                                "latitude": lat,
                                "longitude": lon
                            }
                            
                            all_orgs.append(org_record)
                            
        except Exception as e:
             # Если что-то сломалось на конкретной строке, пропускаем её
             continue
             
    # Превращаем список словарей в красивый DataFrame
    return pd.DataFrame(all_orgs)

# --- Применяем нашу мега-функцию ---

# Создаем отдельную таблицу только для организаций
df_orgs = extract_all_organizations(df)

print(f"Всего найдено организаций: {len(df_orgs)}")
display(df_orgs.head(10))

Всего найдено организаций: 2212


,message_id,org_id,name,address,rating,latitude,longitude
0,7475f82f-d35f-411c-876b-a233f6cd9d3c,72931290140,Konya Büyükşehir Belediye Stadyumu,"Konya, Selçuklu, Parsana Mah.",4.9,37.946209,32.488097
1,2c53e7c8-ae12-462a-96fe-9c5fdbace8c4,157263964577,Takı Dünyası,"İstanbul, Esenyurt, Fevzi Çakmak Cad., 76A",NaN,41.009805,28.694718
2,2c53e7c8-ae12-462a-96fe-9c5fdbace8c4,162694159199,Takı Dünyası,"İstanbul, Fatih, Tahtakale Mah., Yavaşça Şahin...",NaN,41.015818,28.968053
3,11cbac8f-c8cb-4dc6-bb72-a8d4d601f023,1077019784,Tursan Plastik,"İstanbul, Sultanbeyli, Mescidi Aksa Cad.",NaN,40.944813,29.282083
4,666b4d5b-a251-4cc5-96d0-203e4e3a16d0,2216045374,Pursaklar,"Pursaklar, Ankara, Türkiye",NaN,40.105252,32.883900
5,147a975e-810c-4882-93f3-8156aa1879c4,2215917761,Araban,"Araban, Gaziantep, Türkiye",NaN,37.420248,37.762902
6,fadcfca3-0d41-4477-ad3c-1d1e8023d4ca,236729402889,Dubai Beach Konyaaltı,"Antalya, Konyaalti, Akdeniz Boulevard",4.6,36.862495,30.639774
7,66d597f3-d050-42bb-afd0-09c02fb04e53,2215973151,Alpaslan Mah.,"Alpaslan Mah., Bayraklı, İzmir, Türkiye",NaN,38.471737,27.164434
8,e4261939-b3c5-4807-8a06-dbec5c98bd35,1160215885,Kanat Dünyası,"İstanbul, Sultangazi, İsmetpaşa Cad., No/89",4.6,41.106558,28.895696
9,e4261939-b3c5-4807-8a06-dbec5c98bd35,116431030475,Bilican Kebap Restoran,Gazi Mah. İsmet Paşa Cd. No: 53/1A Sultangazi/...,4.0,41.103450,28.897289


In [13]:
import json
import pandas as pd

def extract_all_organizations_with_query(df):
    """
    Проходит по всему DataFrame, извлекает все организации, 
    прикрепляя к каждой из них оригинальный query пользователя.
    """
    all_orgs = []
    
    for index, row in df.iterrows():
        msg_id = row['message_id']
        query = row['query'] # <-- Берем query из текущей строки DataFrame
        response_list = row['model_response_full']
        
        try:
            if isinstance(response_list, list) and len(response_list) > 4:
                payload_str = response_list[4][0]
                data = json.loads(payload_str)
                blocks = data.get("StructuredData", {}).get("BlocksData", [])
                
                for block in blocks:
                    if block.get("ContentType") == "OrganizationList":
                        organizations = block.get("OrganizationList", {}).get("Organizations", [])
                        
                        for org in organizations:
                            
                            # Безопасное извлечение рейтинга
                            rating_val = None
                            if "rating" in org and isinstance(org["rating"], dict):
                                rating_val = org["rating"].get("rating")
                            
                            # Безопасное извлечение координат
                            lat, lon = None, None
                            if "coords" in org and isinstance(org["coords"], dict):
                                lat = org["coords"].get("lat")
                                lon = org["coords"].get("long")
                                
                            # Собираем данные
                            org_record = {
                                "message_id": msg_id,
                                "query": query, # <-- ДОБАВЛЕНО СЮДА
                                "org_id": org.get("id"),
                                "name": org.get("title"),
                                "address": org.get("address"),
                                "rating": rating_val,
                                "latitude": lat,
                                "longitude": lon
                            }
                            
                            all_orgs.append(org_record)
                            
        except Exception as e:
             continue
             
    return pd.DataFrame(all_orgs)

# --- Запускаем и смотрим результат ---

df_orgs_with_query = extract_all_organizations_with_query(df)

print(f"Всего найдено организаций: {len(df_orgs_with_query)}")
display(df_orgs_with_query.head(10))

Всего найдено организаций: 2212


,message_id,query,org_id,name,address,rating,latitude,longitude
0,7475f82f-d35f-411c-876b-a233f6cd9d3c,Konumumdan en yakın rotayı çıkar,72931290140,Konya Büyükşehir Belediye Stadyumu,"Konya, Selçuklu, Parsana Mah.",4.9,37.946209,32.488097
1,2c53e7c8-ae12-462a-96fe-9c5fdbace8c4,Takı Dünyası Nerdedir?,157263964577,Takı Dünyası,"İstanbul, Esenyurt, Fevzi Çakmak Cad., 76A",NaN,41.009805,28.694718
2,2c53e7c8-ae12-462a-96fe-9c5fdbace8c4,Takı Dünyası Nerdedir?,162694159199,Takı Dünyası,"İstanbul, Fatih, Tahtakale Mah., Yavaşça Şahin...",NaN,41.015818,28.968053
3,11cbac8f-c8cb-4dc6-bb72-a8d4d601f023,Tursan fabrikası nerde?,1077019784,Tursan Plastik,"İstanbul, Sultanbeyli, Mescidi Aksa Cad.",NaN,40.944813,29.282083
4,666b4d5b-a251-4cc5-96d0-203e4e3a16d0,pursaklar konum,2216045374,Pursaklar,"Pursaklar, Ankara, Türkiye",NaN,40.105252,32.883900
5,147a975e-810c-4882-93f3-8156aa1879c4,araban ilçesi,2215917761,Araban,"Araban, Gaziantep, Türkiye",NaN,37.420248,37.762902
6,fadcfca3-0d41-4477-ad3c-1d1e8023d4ca,Dubai Beach Lounge Antalya restoran,236729402889,Dubai Beach Konyaaltı,"Antalya, Konyaalti, Akdeniz Boulevard",4.6,36.862495,30.639774
7,66d597f3-d050-42bb-afd0-09c02fb04e53,bak Alparslan mahallesi bayraklı İzmir'de sim,2215973151,Alpaslan Mah.,"Alpaslan Mah., Bayraklı, İzmir, Türkiye",NaN,38.471737,27.164434
8,e4261939-b3c5-4807-8a06-dbec5c98bd35,Lezzet 5 kpdunu mereye giteveğim,1160215885,Kanat Dünyası,"İstanbul, Sultangazi, İsmetpaşa Cad., No/89",4.6,41.106558,28.895696
9,e4261939-b3c5-4807-8a06-dbec5c98bd35,Lezzet 5 kpdunu mereye giteveğim,116431030475,Bilican Kebap Restoran,Gazi Mah. İsmet Paşa Cd. No: 53/1A Sultangazi/...,4.0,41.103450,28.897289


In [14]:
import json
import pandas as pd

def extract_organizations_with_empty_flag(df):
    all_records = []
    
    for index, row in df.iterrows():
        msg_id = row['message_id']
        query = row['query']
        response_list = row['model_response_full']
        
        # По умолчанию считаем, что организаций не найдено
        organizations_found_for_this_query = False
        
        try:
            if isinstance(response_list, list) and len(response_list) > 4:
                payload_str = response_list[4][0]
                data = json.loads(payload_str)
                blocks = data.get("StructuredData", {}).get("BlocksData", [])
                
                for block in blocks:
                    if block.get("ContentType") == "OrganizationList":
                        organizations = block.get("OrganizationList", {}).get("Organizations", [])
                        
                        # Если список организаций не пустой, значит мы их нашли!
                        if organizations:
                            organizations_found_for_this_query = True
                            
                            for org in organizations:
                                rating_val = None
                                if "rating" in org and isinstance(org["rating"], dict):
                                    rating_val = org["rating"].get("rating")
                                
                                lat, lon = None, None
                                if "coords" in org and isinstance(org["coords"], dict):
                                    lat = org["coords"].get("lat")
                                    lon = org["coords"].get("long")
                                    
                                org_record = {
                                    "message_id": msg_id,
                                    "query": query,
                                    "org_found": True, # <-- Флаг: Организация найдена
                                    "org_id": org.get("id"),
                                    "name": org.get("title"),
                                    "address": org.get("address"),
                                    "rating": rating_val,
                                    "latitude": lat,
                                    "longitude": lon
                                }
                                all_records.append(org_record)
                                
        except Exception as e:
             # Если произошла ошибка при парсинге, мы все равно добавим запись ниже
             pass
        
        # Если после разбора всего JSON мы так и не нашли организаций (или была ошибка парсинга)
        # Добавляем строку с запросом, но пустыми данными об организации
        if not organizations_found_for_this_query:
            empty_record = {
                "message_id": msg_id,
                "query": query,
                "org_found": False, # <-- Флаг: Организации НЕТ
                "org_id": None,
                "name": None,
                "address": None,
                "rating": None,
                "latitude": None,
                "longitude": None
            }
            all_records.append(empty_record)
             
    return pd.DataFrame(all_records)

# --- Запускаем и смотрим результат ---

df_final = extract_organizations_with_empty_flag(df)

print(f"Всего строк в результате: {len(df_final)}")
display(df_final.head(10))

Всего строк в результате: 2387


,message_id,query,org_found,org_id,name,address,rating,latitude,longitude
0,7475f82f-d35f-411c-876b-a233f6cd9d3c,Konumumdan en yakın rotayı çıkar,True,72931290140,Konya Büyükşehir Belediye Stadyumu,"Konya, Selçuklu, Parsana Mah.",4.9,37.946209,32.488097
1,2c53e7c8-ae12-462a-96fe-9c5fdbace8c4,Takı Dünyası Nerdedir?,True,157263964577,Takı Dünyası,"İstanbul, Esenyurt, Fevzi Çakmak Cad., 76A",NaN,41.009805,28.694718
2,2c53e7c8-ae12-462a-96fe-9c5fdbace8c4,Takı Dünyası Nerdedir?,True,162694159199,Takı Dünyası,"İstanbul, Fatih, Tahtakale Mah., Yavaşça Şahin...",NaN,41.015818,28.968053
3,0eba2ebf-c5e5-4287-bed0-b76882acda23,Şuan bulunduğum konumdan bursa görükle ye gide...,False,NaN,NaN,NaN,NaN,NaN,NaN
4,ea56cc31-135c-42ca-9f29-0dd20aa4a56d,Milano da gezilecek yerler bana söyler misin?,False,NaN,NaN,NaN,NaN,NaN,NaN
5,c12cf199-14c5-47b5-8ca8-7bc258300c38,karaman adana arası kaç km ve tır ne yakar,False,NaN,NaN,NaN,NaN,NaN,NaN
6,11cbac8f-c8cb-4dc6-bb72-a8d4d601f023,Tursan fabrikası nerde?,True,1077019784,Tursan Plastik,"İstanbul, Sultanbeyli, Mescidi Aksa Cad.",NaN,40.944813,29.282083
7,3d3a476d-03de-4ece-9f9c-bdf66cb196c2,Kızımın adresi,False,NaN,NaN,NaN,NaN,NaN,NaN
8,9e49bd6c-87ac-4b02-a54f-d2e34500c4ef,czech street 126,False,NaN,NaN,NaN,NaN,NaN,NaN
9,666b4d5b-a251-4cc5-96d0-203e4e3a16d0,pursaklar konum,True,2216045374,Pursaklar,"Pursaklar, Ankara, Türkiye",NaN,40.105252,32.883900


In [15]:
import json
import pandas as pd

def extract_organizations_full(df):
    all_records = []
    
    for index, row in df.iterrows():
        msg_id = row['message_id']
        query = row['query']
        response_list = row['model_response_full']
        
        organizations_found_for_this_query = False
        
        try:
            if isinstance(response_list, list) and len(response_list) > 4:
                payload_str = response_list[4][0]
                data = json.loads(payload_str)
                blocks = data.get("StructuredData", {}).get("BlocksData", [])
                
                for block in blocks:
                    if block.get("ContentType") == "OrganizationList":
                        organizations = block.get("OrganizationList", {}).get("Organizations", [])
                        
                        if organizations:
                            organizations_found_for_this_query = True
                            
                            for org in organizations:
                                
                                # --- Извлекаем рейтинг и количество отзывов ---
                                rating_val = None
                                reviews_count_val = None
                                if "rating" in org and isinstance(org["rating"], dict):
                                    rating_val = org["rating"].get("rating")
                                    # Достаем reviewsCount оттуда же, где и сам рейтинг
                                    reviews_count_val = org["rating"].get("reviewsCount")
                                
                                # --- Извлекаем координаты ---
                                lat, lon = None, None
                                if "coords" in org and isinstance(org["coords"], dict):
                                    lat = org["coords"].get("lat")
                                    lon = org["coords"].get("long")
                                
                                # --- Извлекаем тип/категорию организации ---
                                # В твоем JSON это поле "subtitle" (например, "Kuyumcular", "Sanayi kuruluşu")
                                org_type = org.get("subtitle") 
                                
                                # Собираем полные данные
                                org_record = {
                                    "message_id": msg_id,
                                    "query": query,
                                    "org_found": True,
                                    "org_id": org.get("id"),
                                    "name": org.get("title"),
                                    "org_type": org_type,              # <-- ДОБАВЛЕНО: Тип/категория
                                    "address": org.get("address"),
                                    "rating": rating_val,
                                    "reviews_count": reviews_count_val, # <-- ДОБАВЛЕНО: Кол-во отзывов
                                    "latitude": lat,
                                    "longitude": lon
                                }
                                all_records.append(org_record)
                                
        except Exception as e:
             pass
        
        # Если организаций не найдено, добавляем "пустую" строку
        if not organizations_found_for_this_query:
            empty_record = {
                "message_id": msg_id,
                "query": query,
                "org_found": False,
                "org_id": None,
                "name": None,
                "org_type": None,              # <-- Пустое поле для типа
                "address": None,
                "rating": None,
                "reviews_count": None,         # <-- Пустое поле для отзывов
                "latitude": None,
                "longitude": None
            }
            all_records.append(empty_record)
             
    return pd.DataFrame(all_records)

# --- Применяем и любуемся ---

df_final = extract_organizations_full(df)

print(f"Всего строк в результате: {len(df_final)}")
display(df_final.head(10))

Всего строк в результате: 2387


,message_id,query,org_found,org_id,name,org_type,address,rating,reviews_count,latitude,longitude
0,7475f82f-d35f-411c-876b-a233f6cd9d3c,Konumumdan en yakın rotayı çıkar,True,72931290140,Konya Büyükşehir Belediye Stadyumu,Stadyum,"Konya, Selçuklu, Parsana Mah.",4.9,None,37.946209,32.488097
1,2c53e7c8-ae12-462a-96fe-9c5fdbace8c4,Takı Dünyası Nerdedir?,True,157263964577,Takı Dünyası,Kozmetik ve parfümeri mağazaları,"İstanbul, Esenyurt, Fevzi Çakmak Cad., 76A",NaN,None,41.009805,28.694718
2,2c53e7c8-ae12-462a-96fe-9c5fdbace8c4,Takı Dünyası Nerdedir?,True,162694159199,Takı Dünyası,Kuyumcular,"İstanbul, Fatih, Tahtakale Mah., Yavaşça Şahin...",NaN,None,41.015818,28.968053
3,0eba2ebf-c5e5-4287-bed0-b76882acda23,Şuan bulunduğum konumdan bursa görükle ye gide...,False,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
4,ea56cc31-135c-42ca-9f29-0dd20aa4a56d,Milano da gezilecek yerler bana söyler misin?,False,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
5,c12cf199-14c5-47b5-8ca8-7bc258300c38,karaman adana arası kaç km ve tır ne yakar,False,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
6,11cbac8f-c8cb-4dc6-bb72-a8d4d601f023,Tursan fabrikası nerde?,True,1077019784,Tursan Plastik,Plastik ürün üreticileri,"İstanbul, Sultanbeyli, Mescidi Aksa Cad.",NaN,None,40.944813,29.282083
7,3d3a476d-03de-4ece-9f9c-bdf66cb196c2,Kızımın adresi,False,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
8,9e49bd6c-87ac-4b02-a54f-d2e34500c4ef,czech street 126,False,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
9,666b4d5b-a251-4cc5-96d0-203e4e3a16d0,pursaklar konum,True,2216045374,Pursaklar,Area,"Pursaklar, Ankara, Türkiye",NaN,None,40.105252,32.883900


In [16]:
import json
import pandas as pd

def extract_ultimate_data(df):
    all_records = []
    
    for index, row in df.iterrows():
        # --- Данные запроса и ПОЛЬЗОВАТЕЛЯ ---
        msg_id = row['message_id']
        query = row['query']
        user_lat = row.get('latitude')       # Широта пользователя
        user_lon = row.get('longitude')      # Долгота пользователя
        user_region = row.get('region_name') # Регион (например, Istanbul)
        
        response_list = row['model_response_full']
        organizations_found_for_this_query = False
        
        try:
            if isinstance(response_list, list) and len(response_list) > 4:
                payload_str = response_list[4][0]
                data = json.loads(payload_str)
                blocks = data.get("StructuredData", {}).get("BlocksData", [])
                
                for block in blocks:
                    if block.get("ContentType") == "OrganizationList":
                        organizations = block.get("OrganizationList", {}).get("Organizations", [])
                        
                        if organizations:
                            organizations_found_for_this_query = True
                            
                            for org in organizations:
                                rating_val = org.get("rating", {}).get("rating") if isinstance(org.get("rating"), dict) else None
                                reviews_count_val = org.get("rating", {}).get("reviewsCount") if isinstance(org.get("rating"), dict) else None
                                org_lat = org.get("coords", {}).get("lat") if isinstance(org.get("coords"), dict) else None
                                org_lon = org.get("coords", {}).get("long") if isinstance(org.get("coords"), dict) else None
                                org_type = org.get("subtitle") 
                                
                                org_record = {
                                    "message_id": msg_id,
                                    "query": query,
                                    "user_region": user_region, # Откуда пришел запрос
                                    "user_lat": user_lat,       # Координаты юзера
                                    "user_lon": user_lon,       # Координаты юзера
                                    "org_found": True,
                                    "org_id": org.get("id"),
                                    "org_name": org.get("title"),
                                    "org_type": org_type,
                                    "org_address": org.get("address"),
                                    "org_rating": rating_val,
                                    "org_reviews_count": reviews_count_val,
                                    "org_lat": org_lat,         # Координаты места
                                    "org_lon": org_lon          # Координаты места
                                }
                                all_records.append(org_record)
                                
        except Exception as e:
             pass
        
        # Если организаций нет
        if not organizations_found_for_this_query:
            empty_record = {
                "message_id": msg_id,
                "query": query,
                "user_region": user_region,
                "user_lat": user_lat,
                "user_lon": user_lon,
                "org_found": False,
                "org_id": None, "org_name": None, "org_type": None, 
                "org_address": None, "org_rating": None, "org_reviews_count": None, 
                "org_lat": None, "org_lon": None
            }
            all_records.append(empty_record)
             
    return pd.DataFrame(all_records)

# --- Создаем наш идеальный датасет ---
df_final = extract_ultimate_data(df)
display(df_final.head())

,message_id,query,user_region,user_lat,user_lon,org_found,org_id,org_name,org_type,org_address,org_rating,org_reviews_count,org_lat,org_lon
0,7475f82f-d35f-411c-876b-a233f6cd9d3c,Konumumdan en yakın rotayı çıkar,Konya,37.874904,32.492129,True,72931290140,Konya Büyükşehir Belediye Stadyumu,Stadyum,"Konya, Selçuklu, Parsana Mah.",4.9,None,37.946209,32.488097
1,2c53e7c8-ae12-462a-96fe-9c5fdbace8c4,Takı Dünyası Nerdedir?,Istanbul,41.011219,28.978176,True,157263964577,Takı Dünyası,Kozmetik ve parfümeri mağazaları,"İstanbul, Esenyurt, Fevzi Çakmak Cad., 76A",NaN,None,41.009805,28.694718
2,2c53e7c8-ae12-462a-96fe-9c5fdbace8c4,Takı Dünyası Nerdedir?,Istanbul,41.011219,28.978176,True,162694159199,Takı Dünyası,Kuyumcular,"İstanbul, Fatih, Tahtakale Mah., Yavaşça Şahin...",NaN,None,41.015818,28.968053
3,0eba2ebf-c5e5-4287-bed0-b76882acda23,Şuan bulunduğum konumdan bursa görükle ye gide...,Istanbul,41.011219,28.978176,False,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
4,ea56cc31-135c-42ca-9f29-0dd20aa4a56d,Milano da gezilecek yerler bana söyler misin?,Gaziantep,37.062688,37.379510,False,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
